In [1]:
!test -f obs_3day.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/obs_3day.bufr \
             --output-document=obs_3day.bufr
!test -f syn_new.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/syn_new.bufr \
             --output-document=syn_new.bufr

# Generic reader: overview

In [2]:
import pdbufr

## Basic extraction

In [3]:
df = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "data_datetime", "latitude", "longitude", "airTemperatureAt2M", "dewpointTemperatureAt2M"),
)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

50 rows, 6 columns


,ident,latitude,longitude,airTemperatureAt2M,dewpointTemperatureAt2M,data_datetime
0,03894,49.43,-2.60,282.4,274.0,2017-04-25 12:00:00
1,03590,52.12,0.96,281.7,268.4,2017-04-25 12:00:00
2,03379,53.03,-0.50,280.7,269.2,2017-04-25 12:00:00
3,03391,53.09,-0.17,281.0,270.3,2017-04-25 12:00:00
4,03743,51.20,-1.81,281.2,268.4,2017-04-25 12:00:00


### Computed keys

In [4]:
df_wmo = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "WMO_station_id", "latitude", "longitude"),
)
df_wmo.head()

,ident,latitude,longitude,WMO_station_id
0,03894,49.43,-2.60,3894
1,03590,52.12,0.96,3590
2,03379,53.03,-0.50,3379
3,03391,53.09,-0.17,3391
4,03743,51.20,-1.81,3743


## Hierarchical extraction

In [5]:
df_cloud = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude", "cloudType", "heightOfBaseOfCloud"),
)
print(f"{len(df_cloud)} rows, {len(df_cloud.columns)} columns")
df_cloud.head(8)

350 rows, 5 columns


,ident,latitude,longitude,heightOfBaseOfCloud,cloudType
0,03894,49.43,-2.60,1250.0,32.0
1,03894,49.43,-2.60,1250.0,20.0
2,03894,49.43,-2.60,1250.0,11.0
3,03894,49.43,-2.60,1250.0,8.0
4,03894,49.43,-2.60,980.0,NaN
5,03894,49.43,-2.60,NaN,NaN
6,03894,49.43,-2.60,NaN,NaN
7,03590,52.12,0.96,1250.0,31.0


`obs_3day.bufr` has 50 messages but the result above has 350 rows because each station reports up to 7 cloud layers. Stations reporting fewer layers simply end up with fewer rows, and the unused layers are filled with `NaN`.

## The `required_columns` parameter

### Default behaviour (`required_columns=True`)

By default all the keys in `columns` are required: whenever one of them is missing for a given hierarchy branch the whole record is dropped. Here no branch has both the temperature keys and the wind keys at the same time, so nothing is returned:

In [6]:
columns = (
    "stationNumber",
    "airTemperature",
    "dewpointTemperature",
    "windDirection",
    "windSpeed",
)
filters = {"heightOfSensorAboveLocalGroundOrDeckOfMarinePlatform": slice(1, 20)}
df_rc_true = pdbufr.read_bufr(
    "syn_new.bufr",
    columns=columns,
    filters=filters,
)
print(f"{len(df_rc_true)} rows")
df_rc_true

0 rows


""


### `required_columns=False`

No column is required, so a record is produced for every branch and missing values become `NaN`:

In [7]:
df_rc_false = pdbufr.read_bufr(
    "syn_new.bufr",
    columns=columns,
    filters=filters,
    required_columns=False,
)
print(f"{len(df_rc_false)} rows")
df_rc_false

14 rows


,stationNumber,airTemperature,dewpointTemperature,windDirection,windSpeed
0,948,300.45,295.15,NaN,NaN
1,948,NaN,NaN,100.0,1.6
2,948,NaN,NaN,NaN,NaN
3,948,NaN,NaN,NaN,NaN
4,948,NaN,NaN,NaN,NaN
5,766,269.25,263.55,NaN,NaN
6,766,NaN,NaN,NaN,NaN
7,766,NaN,NaN,NaN,NaN
8,766,NaN,NaN,NaN,NaN
9,766,NaN,NaN,NaN,NaN


### `required_columns` as a list

Passing an explicit list of keys requires only those keys to be present, independently of `columns`. Requiring just `stationNumber`, which is always present, reproduces the `required_columns=False` result here:

In [8]:
df_rc_list = pdbufr.read_bufr(
    "syn_new.bufr",
    columns=columns,
    filters=filters,
    required_columns=["stationNumber"],
)
print(f"{len(df_rc_list)} rows")
assert df_rc_list.shape == df_rc_false.shape

14 rows


## Filters

### Single value and list filters

Select a single station or a list of stations by WMO block+station number:

In [9]:
# Single station
df_st = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude", "airTemperatureAt2M"),
    filters={"stationNumber": 894},
)
df_st

,ident,latitude,longitude,airTemperatureAt2M
0,03894,49.43,-2.6,282.4


In [10]:
# Multiple stations - pass a list
df_sts = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude", "airTemperatureAt2M"),
    filters={"stationNumber": [894, 103]},
)
df_sts

,ident,latitude,longitude,airTemperatureAt2M
0,03894,49.43,-2.60,282.4
1,07103,48.04,-4.73,NaN


### Slice filter

Select the first three messages using the built-in `count` key (1-based message counter):

In [11]:
df_slice = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude"),
    filters={"count": slice(1, 4)},
)
df_slice

,ident,latitude,longitude
0,03894,49.43,-2.60
1,03590,52.12,0.96
2,03379,53.03,-0.50
3,03391,53.09,-0.17


### Callable filter

A callable condition can express arbitrary logic. Here we keep only the stations where the 2m temperature exceeds 283 K:

In [12]:
df_cb = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude", "airTemperatureAt2M"),
    filters={"airTemperatureAt2M": lambda t: t > 283},
)
df_cb

,ident,latitude,longitude,airTemperatureAt2M
0,07005,50.14,1.83,283.2
1,07117,48.83,-3.47,284.5
2,07100,48.47,-5.06,283.3
3,07107,48.68,-4.33,283.3


### Combining conditions (geographical filter)

Conditions on several keys are combined with a logical AND. Combining `slice` filters on `latitude` and `longitude` gives a geographical filter:

In [13]:
df_geo = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "latitude", "longitude"),
    filters={"latitude": slice(50, 55), "longitude": slice(-5, 5)},
)
print(f"{len(df_geo)} rows")
df_geo.head()

24 rows


,ident,latitude,longitude
0,03590,52.12,0.96
1,03379,53.03,-0.50
2,03391,53.09,-0.17
3,03743,51.20,-1.81
4,03749,51.15,-1.57


### Computed-key filter

Computed keys can also be used in filters. `WMO_station_id` matches when `blockNumber * 1000 + stationNumber` equals the given value:

In [14]:
df_wmo_f = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "WMO_station_id", "latitude", "longitude"),
    filters={"WMO_station_id": [3894, 7103]},
)
df_wmo_f

,ident,latitude,longitude,WMO_station_id
0,03894,49.43,-2.60,3894
1,07103,48.04,-4.73,7103


## The `prefilter_headers` parameter

In [15]:
df_prefilter = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=("ident", "typicalDate", "typicalTime", "latitude", "longitude"),
    filters={"bufrHeaderCentre": 98},
    prefilter_headers=True,
)
print(f"{len(df_prefilter)} rows")
df_prefilter.head()

50 rows


,typicalDate,typicalTime,ident,latitude,longitude
0,20170425,120000,03894,49.43,-2.60
1,20170425,120000,03590,52.12,0.96
2,20170425,120000,03379,53.03,-0.50
3,20170425,120000,03391,53.09,-0.17
4,20170425,120000,03743,51.20,-1.81
